# 03. Scheduler、KV Memory Pool 与执行路径

> 面向即将加入 SGLang 开源社区的开发者。建议从仓库根目录启动 Jupyter，
> 或者在 notebook 第一格运行路径检查。本文以本地源码为主，线上文档为索引。
> Markdown 中的源码摘录会插入 `# 注：...` 行内讲解；可执行代码 cell 则保持可运行。


In [ ]:
from pathlib import Path
import ast, inspect, re, textwrap

def find_repo_root(start=None):
    p = Path(start or Path.cwd()).resolve()
    for candidate in [p, *p.parents]:
        if (candidate / "python" / "sglang").exists() and (candidate / "docs").exists():
            return candidate
    raise RuntimeError("没有找到 SGLang 仓库根目录，请从仓库内启动 notebook。")

ROOT = find_repo_root()
print(ROOT)

def read_rel(path):
    return (ROOT / path).read_text()

def show_lines(path, start, end):
    lines = read_rel(path).splitlines()
    for i in range(start, min(end, len(lines)) + 1):
        print(f"{i:4d}: {lines[i-1]}")

def find_defs(path, names=None):
    tree = ast.parse(read_rel(path))
    rows = []
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):
            if names is None or node.name in names:
                rows.append((node.lineno, type(node).__name__, node.name))
    return sorted(rows)


## Scheduler 关心的不是“一个请求”，而是“资源可行的下一批请求”

每轮循环里 scheduler 都要在这些约束之间取平衡：

- waiting queue 中哪些请求可以进入 prefill。
- running batch 中哪些请求继续 decode。
- KV cache 还有多少空位，可以驱逐多少，哪些节点被锁住。
- grammar 是否已编译完成。
- speculative decoding 是否需要 draft/verify。
- overlap schedule 是否允许 CPU 调度和 GPU 执行重叠。

这就是 SGLang 的吞吐核心：连续 batching、prefix cache、paged KV、chunked prefill 都在 scheduler 汇合。


In [ ]:
for path in [
    "python/sglang/srt/managers/scheduler.py",
    "python/sglang/srt/managers/schedule_batch.py",
    "python/sglang/srt/mem_cache/memory_pool.py",
    "python/sglang/srt/model_executor/forward_batch_info.py",
    "python/sglang/srt/model_executor/model_runner.py",
]:
    print("\n==", path)
    for row in find_defs(path, names={"Scheduler", "Req", "ScheduleBatch", "ReqToTokenPool", "ForwardBatch", "ModelRunner"}):
        print(row)


## `Req`、`ScheduleBatch`、`ForwardBatch`

三个对象对应三个抽象层：

- `Req`：单个请求的 CPU 状态，包含 input ids、output ids、sampling params、prefix hit、grammar、finish reason。
- `ScheduleBatch`：scheduler 选出的一个 batch，仍以 Python/CPU 状态为主，但已经决定了 extend/decode、KV 分配等。
- `ForwardBatch`：model runner 消费的 tensor 化执行元数据，attention backend 依赖它知道 seq_lens、cache loc、forward mode。


### `Req.__init__`：单个请求的“账本”

```python
# python/sglang/srt/managers/schedule_batch.py:666-745
 666: class Req(ReqDllmMixin):
      # 注：类定义：`Req` 是这一段的状态/行为边界；先看字段，再看哪些方法会改字段。
 667:     """The input and output status of a request."""
 668: 
 669:     def __init__(
      # 注：函数定义：`__init__` 是调用边界；参数决定它从上游接收哪些状态，返回值决定下游能看到什么。
 670:         self,
 671:         rid: str,
 672:         origin_input_text: str,
 673:         origin_input_ids: array[int],
 674:         sampling_params: SamplingParams,
 675:         return_logprob: bool = False,
 676:         top_logprobs_num: int = 0,
 677:         dllm_config: Optional[DllmConfig] = None,
 678:         token_ids_logprob: List[int] = None,
 679:         stream: bool = False,
 680:         origin_input_ids_unpadded: Optional[array[int]] = None,
 681:         lora_id: Optional[str] = None,
 682:         input_embeds: Optional[List[List[float]]] = None,
 683:         positional_embed_overrides: Optional[PositionalEmbeds] = None,
 684:         token_type_ids: List[int] = None,
 685:         session: Optional[Session] = None,
 686:         custom_logit_processor: Optional[str] = None,
 687:         require_reasoning: bool = False,
 688:         return_hidden_states: bool = False,
 689:         return_routed_experts: bool = False,
 690:         routed_experts_start_len: int = 0,
 691:         return_indexer_topk: bool = False,
 692:         eos_token_ids: Optional[Set[int]] = None,
 693:         bootstrap_host: Optional[str] = None,
 694:         bootstrap_port: Optional[int] = None,
 695:         bootstrap_room: Optional[int] = None,
 696:         disagg_mode: Optional[DisaggregationMode] = None,
 697:         routed_dp_rank: Optional[int] = None,
 698:         disagg_prefill_dp_rank: Optional[int] = None,
 699:         vocab_size: Optional[int] = None,
 700:         priority: Optional[int] = None,
 701:         metrics_collector: Optional[SchedulerMetricsCollector] = None,
 702:         extra_key: Optional[str] = None,
 703:         routing_key: Optional[str] = None,
 704:         dimensions: Optional[int] = None,
 705:         http_worker_ipc: Optional[str] = None,
 706:         time_stats: Optional[
      # 注：字段/变量声明：`time_stats` 带有类型信息；它描述这个对象或阶段会保存的状态形状。
 707:             Union[APIServerReqTimeStats, DPControllerReqTimeStats]
 708:         ] = None,
 709:         return_pooled_hidden_states: bool = False,
 710:         multi_item_delimiter_indices: Optional[List[int]] = None,
 711:     ):
 712:         # Input and output info
 713:         self.rid = rid
      # 注：成员变量写入：`self.rid` 会留在对象生命周期中，后续方法可能依赖这份状态。
 714:         self.origin_input_ids = origin_input_ids
      # 注：成员变量写入：`self.origin_input_ids` 保存 token 相关状态，后续长度计算、cache key 或采样都会读取它。
 715:         self.origin_input_ids_unpadded = (
      # 注：成员变量写入：`self.origin_input_ids_unpadded` 保存 token 相关状态，后续长度计算、cache key 或采样都会读取它。
 716:             origin_input_ids_unpadded
 717:             if origin_input_ids_unpadded
      # 注：分支判断：只有满足 `origin_input_ids_unpadded` 时才进入该路径；这通常是在区分部署模式、请求类型或资源状态。
 718:             else self.origin_input_ids
 719:         )  # Before image padding
 720:         # Each decode stage's output ids. Append-only by contract:
 721:         # _refresh_fill_ids infers how many output tokens are already in
 722:         # full_untruncated_fill_ids from lengths alone, so in-place rewrites
 723:         # that preserve length would silently corrupt fill_ids.
 724:         self.output_ids = array("q")
      # 注：成员变量写入：`self.output_ids` 保存 token 相关状态，后续长度计算、cache key 或采样都会读取它。
 725:         # Full untruncated sequence: origin + output (+ DLLM mask block).
 726:         # Kept in sync by _refresh_fill_ids; admission only updates fill_len,
 727:         # never mutates this array's length.
 728:         self.full_untruncated_fill_ids = array("q")
      # 注：成员变量写入：`self.full_untruncated_fill_ids` 保存 token 相关状态，后续长度计算、cache key 或采样都会读取它。
 729:         self.fill_len: int = 0
      # 注：成员变量写入：`self.fill_len` 会留在对象生命周期中，后续方法可能依赖这份状态。
 730: 
 731:         self.session = session
      # 注：成员变量写入：`self.session` 会留在对象生命周期中，后续方法可能依赖这份状态。
 732:         self.input_embeds = input_embeds
      # 注：成员变量写入：`self.input_embeds` 会留在对象生命周期中，后续方法可能依赖这份状态。
 733:         self.positional_embed_overrides = positional_embed_overrides
      # 注：成员变量写入：`self.positional_embed_overrides` 会留在对象生命周期中，后续方法可能依赖这份状态。
 734:         self.multi_item_delimiter_indices = multi_item_delimiter_indices
      # 注：成员变量写入：`self.multi_item_delimiter_indices` 会留在对象生命周期中，后续方法可能依赖这份状态。
 735: 
 736:         # For req-level memory management
 737:         self.kv_committed_len = 0
      # 注：成员变量写入：`self.kv_committed_len` 会留在对象生命周期中，后续方法可能依赖这份状态。
 738:         self.kv_allocated_len = 0
      # 注：成员变量写入：`self.kv_allocated_len` 会留在对象生命周期中，后续方法可能依赖这份状态。
 739:         self.kv_committed_freed = False
      # 注：成员变量写入：`self.kv_committed_freed` 会留在对象生命周期中，后续方法可能依赖这份状态。
 740:         self.kv_overallocated_freed = False
      # 注：成员变量写入：`self.kv_overallocated_freed` 会留在对象生命周期中，后续方法可能依赖这份状态。
 741: 
 742:         # for corss-endoder model
 743:         self.token_type_ids = token_type_ids
      # 注：成员变量写入：`self.token_type_ids` 保存 token 相关状态，后续长度计算、cache key 或采样都会读取它。
 744: 
 745:         # The length of KV that have been removed in swa cache.
```

**读这段时抓住：**

- `origin_input_ids` 是 prompt，`output_ids` 是 append-only 的生成结果；许多后续长度计算假设 output 只追加不重写。
- `kv_committed_len` / `kv_allocated_len` 区分已经提交给请求语义的 KV 和临时/预分配 KV。
- `extra_key`、`lora_id`、`routing_key` 等字段决定调度、cache 隔离和分布式路由，不只是 API 元数据。


### `ScheduleBatch`：scheduler 侧 batch 同时拿着资源和请求

```python
# python/sglang/srt/managers/schedule_batch.py:1671-1738
1671: class ScheduleBatch(ScheduleBatchDisaggregationDecodeMixin):
      # 注：类定义：`ScheduleBatch` 是这一段的状态/行为边界；先看字段，再看哪些方法会改字段。
1672:     """Store all information of a batch on the scheduler."""
1673: 
1674:     # === Core: request list (ForwardBatch derives lora_ids / rids / grammars / positions from it) ===
1675:     reqs: List[Req]
      # 注：字段/变量声明：`reqs` 带有类型信息；它描述这个对象或阶段会保存的状态形状。
1676: 
1677:     # === Global config and shared resources (engine-lifetime; identical across batches) ===
1678:     # Memory pool and cache
1679:     req_to_token_pool: ReqToTokenPool = None
      # 注：字段/变量声明：`req_to_token_pool` 带有类型信息；它描述这个对象或阶段会保存的状态形状。
1680:     token_to_kv_pool_allocator: BaseTokenToKVPoolAllocator = None
      # 注：字段/变量声明：`token_to_kv_pool_allocator` 带有类型信息；它描述这个对象或阶段会保存的状态形状。
1681:     tree_cache: BasePrefixCache = None
      # 注：字段/变量声明：`tree_cache` 带有类型信息；它描述这个对象或阶段会保存的状态形状。
1682: 
1683:     # Batch configs
1684:     model_config: ModelConfig = None
      # 注：局部状态绑定：`model_config` 保存配置快照，下面的分支通常会基于它选择执行路径。
1685:     enable_overlap: bool = False
      # 注：字段/变量声明：`enable_overlap` 带有类型信息；它描述这个对象或阶段会保存的状态形状。
1686: 
1687:     # Device
1688:     device: str = "cuda"
      # 注：字段/变量声明：`device` 带有类型信息；它描述这个对象或阶段会保存的状态形状。
1689: 
1690:     # HiSparse (engine-level coordinator ref, same across batches)
1691:     hisparse_coordinator: Optional[HiSparseCoordinator] = None
      # 注：字段/变量声明：`hisparse_coordinator` 带有类型信息；它描述这个对象或阶段会保存的状态形状。
1692: 
1693:     # === Batch-variant scheduler state (per-batch; not read by ForwardBatch) ===
1694:     # Tell whether the current running batch is full so that we can skip
1695:     # the check of whether to prefill new requests.
1696:     # This is an optimization to reduce the overhead of the prefill check.
1697:     batch_is_full: bool = False
      # 注：字段/变量声明：`batch_is_full` 带有类型信息；它描述这个对象或阶段会保存的状态形状。
1698: 
1699:     # For chunked prefill in PP
1700:     chunked_req: Optional[Req] = None
      # 注：字段/变量声明：`chunked_req` 带有类型信息；它描述这个对象或阶段会保存的状态形状。
1701:     chunked_req_next_prompt_token: Optional[int] = None
      # 注：字段/变量声明：`chunked_req_next_prompt_token` 带有类型信息；它描述这个对象或阶段会保存的状态形状。
1702:     contains_last_prefill_chunk: bool = True
      # 注：字段/变量声明：`contains_last_prefill_chunk` 带有类型信息；它描述这个对象或阶段会保存的状态形状。
1703: 
1704:     # For DP attention
1705:     inner_idle_batch: Optional[ScheduleBatch] = None
      # 注：字段/变量声明：`inner_idle_batch` 带有类型信息；它描述这个对象或阶段会保存的状态形状。
1706:     # Decode requests carried alongside a chunked-prefill batch
1707:     decoding_reqs: List[Req] = None
      # 注：字段/变量声明：`decoding_reqs` 带有类型信息；它描述这个对象或阶段会保存的状态形状。
1708: 
1709:     # For split prefill
1710:     split_index: int = 0
      # 注：字段/变量声明：`split_index` 带有类型信息；它描述这个对象或阶段会保存的状态形状。
1711:     split_prefill_finished: bool = False
      # 注：字段/变量声明：`split_prefill_finished` 带有类型信息；它描述这个对象或阶段会保存的状态形状。
1712:     split_forward_count: int = 1
      # 注：字段/变量声明：`split_forward_count` 带有类型信息；它描述这个对象或阶段会保存的状态形状。
1713:     split_forward_batch: ForwardBatch = None
      # 注：字段/变量声明：`split_forward_batch` 带有类型信息；它描述这个对象或阶段会保存的状态形状。
1714: 
1715:     # CPU mirror of req_pool_indices; schedule-path only (used in overlap_utils,
1716:     # not read by ForwardBatch), stale in spec draft window
1717:     req_pool_indices_cpu: torch.Tensor = None  # shape: [b], int64
      # 注：字段/变量声明：`req_pool_indices_cpu` 带有类型信息；它描述这个对象或阶段会保存的状态形状。
1718: 
1719:     # Forward-pass metrics
1720:     fpm_start_time: float = 0.0
      # 注：字段/变量声明：`fpm_start_time` 带有类型信息；它描述这个对象或阶段会保存的状态形状。
1721: 
1722:     # hicache pointer for synchronizing data loading from CPU to GPU
1723:     hicache_consumer_index: int = -1
      # 注：字段/变量声明：`hicache_consumer_index` 带有类型信息；它描述这个对象或阶段会保存的状态形状。
1724: 
1725:     # Metrics
1726:     dp_cooperation_info: Optional[DPCooperationInfo] = None
      # 注：字段/变量声明：`dp_cooperation_info` 带有类型信息；它描述这个对象或阶段会保存的状态形状。
1727:     prefill_stats: Optional[PrefillStats] = None
      # 注：字段/变量声明：`prefill_stats` 带有类型信息；它描述这个对象或阶段会保存的状态形状。
1728:     forward_iter: Optional[int] = None
      # 注：字段/变量声明：`forward_iter` 带有类型信息；它描述这个对象或阶段会保存的状态形状。
1729: 
1730:     # === GPU tensors crossing to ForwardBatch (clone targets for stream isolation) ===
1731:     # Batched arguments to model runner
1732:     input_ids: torch.Tensor = None  # shape: [b], int64
      # 注：字段/变量声明：`input_ids` 带有类型信息；它描述这个对象或阶段会保存的状态形状。
1733:     # Staging consumed by resolve_forward_inputs (prefill H2D / mixed gather).
1734:     prefill_input_ids_cpu: Optional[torch.Tensor] = None
      # 注：字段/变量声明：`prefill_input_ids_cpu` 带有类型信息；它描述这个对象或阶段会保存的状态形状。
1735:     mix_running_indices: Optional[torch.Tensor] = None
      # 注：字段/变量声明：`mix_running_indices` 带有类型信息；它描述这个对象或阶段会保存的状态形状。
1736:     input_embeds: torch.Tensor = None  # shape: [b, hidden_size], float32
      # 注：字段/变量声明：`input_embeds` 带有类型信息；它描述这个对象或阶段会保存的状态形状。
1737: 
1738:     # Token replacement embeddings and absolute positions (optional).
```

**读这段时抓住：**

- `reqs` 是高层请求列表；`req_to_token_pool`、`token_to_kv_pool_allocator`、`tree_cache` 是共享资源引用。
- batch-variant 字段记录 chunked prefill、split prefill、HiCache consumer index、metrics 等调度状态。
- `input_ids`、`req_pool_indices` 等 tensor 字段是跨到 `ForwardBatch` 的桥，不是简单 Python list。


### `ForwardBatch`：attention backend 真正读取的执行快照

```python
# python/sglang/srt/model_executor/forward_batch_info.py:322-388
 322: class ForwardBatch(ForwardBatchDeepSeekMHAMixin):
      # 注：类定义：`ForwardBatch` 是这一段的状态/行为边界；先看字段，再看哪些方法会改字段。
 323:     """Store all inputs of a forward pass."""
 324: 
 325:     # === Required core inputs (no default; input_ids / req_pool_indices / seq_lens / out_cache_loc are borrowed from ScheduleBatch) ===
 326:     # The forward mode
 327:     forward_mode: ForwardMode
      # 注：字段/变量声明：`forward_mode` 带有类型信息；它描述这个对象或阶段会保存的状态形状。
 328:     # The batch size
 329:     batch_size: int
      # 注：字段/变量声明：`batch_size` 带有类型信息；它描述这个对象或阶段会保存的状态形状。
 330:     # The input ids
 331:     input_ids: torch.Tensor
      # 注：字段/变量声明：`input_ids` 带有类型信息；它描述这个对象或阶段会保存的状态形状。
 332:     # The indices of requests in the req_to_token_pool
 333:     req_pool_indices: torch.Tensor
      # 注：字段/变量声明：`req_pool_indices` 带有类型信息；它描述这个对象或阶段会保存的状态形状。
 334:     # The sequence length
 335:     seq_lens: torch.Tensor
      # 注：字段/变量声明：`seq_lens` 带有类型信息；它描述这个对象或阶段会保存的状态形状。
 336:     # The indices of output tokens in the token_to_kv_pool
 337:     out_cache_loc: torch.Tensor
      # 注：字段/变量声明：`out_cache_loc` 带有类型信息；它描述这个对象或阶段会保存的状态形状。
 338:     # The sum of all sequence lengths
 339:     seq_lens_sum: int
      # 注：字段/变量声明：`seq_lens_sum` 带有类型信息；它描述这个对象或阶段会保存的状态形状。
 340: 
 341:     # === Borrowed from ScheduleBatch: GPU tensors (cross-stream; clone targets for stream isolation) ===
 342:     # FIXME(lsyin): these are currently aliased by reference from ScheduleBatch. Once
 343:     # they are cloned/relayed into FB-owned copies at the boundary, move them out of
 344:     # "Borrowed" into a dedicated "Forward-resolved snapshot" group.
 345:     # The original sequence length without being chunked. Qwen-1M related.
 346:     orig_seq_lens: Optional[torch.Tensor] = None
      # 注：字段/变量声明：`orig_seq_lens` 带有类型信息；它描述这个对象或阶段会保存的状态形状。
 347: 
 348:     # DSV4-NPU only: per-pool slot bundle from DSV4NPUTokenToKVPoolAllocator,
 349:     # consumed by the Ascend backend for PA_ND block tables. None elsewhere.
 350:     out_cache_loc_dsv4: Optional[DSV4OutCacheLoc] = None
      # 注：字段/变量声明：`out_cache_loc_dsv4` 带有类型信息；它描述这个对象或阶段会保存的状态形状。
 351:     # The indices to track mamba state with
 352:     mamba_track_indices: Optional[torch.Tensor] = None  # shape: [b], int64
      # 注：字段/变量声明：`mamba_track_indices` 带有类型信息；它描述这个对象或阶段会保存的状态形状。
 353:     # The mask to track mamba state if needed
 354:     mamba_track_mask: Optional[torch.Tensor] = None  # shape: [b], bool
      # 注：字段/变量声明：`mamba_track_mask` 带有类型信息；它描述这个对象或阶段会保存的状态形状。
 355:     # The seqlens to track mamba state if masked, prefill only.
 356:     mamba_track_seqlens: Optional[torch.Tensor] = None  # shape: [b], int64
      # 注：字段/变量声明：`mamba_track_seqlens` 带有类型信息；它描述这个对象或阶段会保存的状态形状。
 357:     # Deferred mamba init ops: COW pairs and clear indices (performed on forward stream)
 358:     mamba_cow_src_indices: Optional[torch.Tensor] = None
      # 注：字段/变量声明：`mamba_cow_src_indices` 带有类型信息；它描述这个对象或阶段会保存的状态形状。
 359:     mamba_cow_dst_indices: Optional[torch.Tensor] = None
      # 注：字段/变量声明：`mamba_cow_dst_indices` 带有类型信息；它描述这个对象或阶段会保存的状态形状。
 360:     mamba_clear_indices: Optional[torch.Tensor] = None
      # 注：字段/变量声明：`mamba_clear_indices` 带有类型信息；它描述这个对象或阶段会保存的状态形状。
 361: 
 362:     # For input embeddings
 363:     input_embeds: Optional[torch.Tensor] = None
      # 注：字段/变量声明：`input_embeds` 带有类型信息；它描述这个对象或阶段会保存的状态形状。
 364:     # For token embedding overrides (sparse replacement at specific positions)
 365:     replace_embeds: Optional[torch.Tensor] = None
      # 注：字段/变量声明：`replace_embeds` 带有类型信息；它描述这个对象或阶段会保存的状态形状。
 366:     replace_positions: Optional[torch.Tensor] = None
      # 注：字段/变量声明：`replace_positions` 带有类型信息；它描述这个对象或阶段会保存的状态形状。
 367: 
 368:     # For cross-encoder model
 369:     token_type_ids: Optional[torch.Tensor] = None
      # 注：字段/变量声明：`token_type_ids` 带有类型信息；它描述这个对象或阶段会保存的状态形状。
 370: 
 371:     # Encoder-decoder device tensors
 372:     encoder_lens: Optional[torch.Tensor] = None
      # 注：字段/变量声明：`encoder_lens` 带有类型信息；它描述这个对象或阶段会保存的状态形状。
 373:     encoder_out_cache_loc: Optional[torch.Tensor] = None
      # 注：字段/变量声明：`encoder_out_cache_loc` 带有类型信息；它描述这个对象或阶段会保存的状态形状。
 374: 
 375:     # === Borrowed from ScheduleBatch: config / flags (by-value) ===
 376:     # For logprob
 377:     return_logprob: bool = False
      # 注：字段/变量声明：`return_logprob` 带有类型信息；它描述这个对象或阶段会保存的状态形状。
 378:     # Whether this batch is prefill-only (no token generation needed)
 379:     is_prefill_only: bool = False
      # 注：字段/变量声明：`is_prefill_only` 带有类型信息；它描述这个对象或阶段会保存的状态形状。
 380:     spec_algorithm: SpeculativeAlgorithm = None
      # 注：字段/变量声明：`spec_algorithm` 带有类型信息；它描述这个对象或阶段会保存的状态形状。
 381:     # For matryoshka embeddings
 382:     dimensions: Optional[list[int]] = None
      # 注：字段/变量声明：`dimensions` 带有类型信息；它描述这个对象或阶段会保存的状态形状。
 383:     # Whether to return pooled hidden states (pre-head transformer output)
 384:     return_pooled_hidden_states: bool = False
      # 注：字段/变量声明：`return_pooled_hidden_states` 带有类型信息；它描述这个对象或阶段会保存的状态形状。
 385: 
 386:     # For DP attention
 387:     is_extend_in_batch: bool = False
      # 注：字段/变量声明：`is_extend_in_batch` 带有类型信息；它描述这个对象或阶段会保存的状态形状。
 388:     # Mirrors ScheduleBatch.all_extend_in_batch; kept for downstream forks.
```

**读这段时抓住：**

- `forward_mode`、`seq_lens`、`out_cache_loc` 是 attention backend 决定 prefill/decode/verify 行为的核心。
- 许多字段从 `ScheduleBatch` 借用而非复制，说明 stream isolation 和生命周期管理非常敏感。
- speculative、DP attention、Mamba、encoder-decoder 都在这里挂执行元数据，所以新增执行路径往往会扩展 ForwardBatch。


In [ ]:
for name in ["Req", "ScheduleBatch"]:
    rows = find_defs("python/sglang/srt/managers/schedule_batch.py", {name})
    print(name, rows)
for name in ["ForwardBatch", "ForwardMode"]:
    rows = find_defs("python/sglang/srt/model_executor/forward_batch_info.py", {name})
    print(name, rows)


In [ ]:
show_lines("python/sglang/srt/managers/schedule_batch.py", 1, 35)
show_lines("python/sglang/srt/managers/schedule_batch.py", 35, 75)


## KV memory pool：为什么需要两层映射

LLM serving 中 KV cache 是巨大的 `[layers, tokens, heads, dim]` 存储。SGLang 会把请求逻辑位置映射到物理 KV slot：

- `ReqToTokenPool`：请求维度的 token position -> KV slot index。
- `TokenToKVPool` / allocator：KV slot index -> 每层 K/V tensor 中的实际位置。

这样 decode 时新增 token 只需追加 slot，prefix cache 命中时可以直接复用已有 slot indices。


In [ ]:
for path in [
    "python/sglang/srt/mem_cache/memory_pool.py",
    "python/sglang/srt/mem_cache/allocator/base.py",
    "python/sglang/srt/mem_cache/common.py",
]:
    print("\n==", path)
    for lineno, kind, name in find_defs(path):
        if "Pool" in name or name in {"alloc_for_extend", "alloc_for_decode", "release_kv_cache", "evict_from_tree_cache"}:
            print(f"{lineno:4d} {kind:16s} {name}")


## prefill 与 decode 的调度差异

- **Prefill / extend**：一次处理 prompt 或 chunk，可能写入大量 KV；受 `max_prefill_tokens`、chunked prefill、prefix hit、grammar readiness 影响。
- **Decode**：每个 running request 通常前进一步；关注 `max_running_requests`、KV reserve、sampling、finish。

Speculative decoding 会把 decode 变复杂：一次可能 draft 多个 token，再由 target verify 接受若干 token。


In [ ]:
show_lines("python/sglang/srt/managers/scheduler.py", 2735, 2795)
show_lines("python/sglang/srt/managers/scheduler.py", 3178, 3225)


### `get_new_batch_prefill`：新请求进入 GPU 前的门禁

```python
# python/sglang/srt/managers/scheduler.py:2735-2795
2735:     def get_new_batch_prefill(self) -> Optional[ScheduleBatch]:
      # 注：函数定义：`get_new_batch_prefill` 是调用边界；参数决定它从上游接收哪些状态，返回值决定下游能看到什么。
2736:         prefill_delayer_single_pass = None
      # 注：局部状态绑定：`prefill_delayer_single_pass` 保存本阶段的中间结果，后续几行通常会立即消费它。
2737:         if self.prefill_delayer:
      # 注：分支判断：只有满足 `self.prefill_delayer` 时才进入该路径；这通常是在区分部署模式、请求类型或资源状态。
2738:             # Get max usage across all pools for prefill delay decision
2739:             max_pool_usage = (
      # 注：局部状态绑定：`max_pool_usage` 保存本阶段的中间结果，后续几行通常会立即消费它。
2740:                 self.pool_stats_observer.get_pool_stats().get_max_pool_usage()
      # 注：成员函数调用：`self.pool_stats_observer.get_pool_stats` 会读取或更新当前对象状态，建议继续查看该方法定义。
2741:             )
2742:             prefill_delayer_single_pass = PrefillDelayerSinglePassExecutor(
      # 注：局部状态绑定：`prefill_delayer_single_pass` 保存本阶段的中间结果，后续几行通常会立即消费它。
2743:                 self.prefill_delayer, token_usage=max_pool_usage
      # 注：成员变量写入：`self.prefill_delayer, token_usage` 保存 token 相关状态，后续长度计算、cache key 或采样都会读取它。
2744:             )
2745: 
2746:         ret = self._get_new_batch_prefill_raw(
      # 注：局部状态绑定：`ret` 保存本阶段的中间结果，后续几行通常会立即消费它。
2747:             prefill_delayer_single_pass=prefill_delayer_single_pass
      # 注：局部状态绑定：`prefill_delayer_single_pass` 保存本阶段的中间结果，后续几行通常会立即消费它。
2748:         )
2749: 
2750:         if self.prefill_delayer:
      # 注：分支判断：只有满足 `self.prefill_delayer` 时才进入该路径；这通常是在区分部署模式、请求类型或资源状态。
2751:             prefill_delayer_single_pass.finalize(actual_prefill=ret is not None)
      # 注：局部状态绑定：`prefill_delayer_single_pass.finalize(actual_prefill` 保存本阶段的中间结果，后续几行通常会立即消费它。
2752: 
2753:         return ret
      # 注：返回出口：把本阶段整理出的状态交给调用方；读调用链时要回到上层看它如何被消费。
2754: 
2755:     def _get_new_batch_prefill_raw(
      # 注：函数定义：`_get_new_batch_prefill_raw` 是调用边界；参数决定它从上游接收哪些状态，返回值决定下游能看到什么。
2756:         self, prefill_delayer_single_pass: Optional[PrefillDelayerSinglePassExecutor]
2757:     ) -> Optional[ScheduleBatch]:
2758:         # Check if the grammar is ready in the grammar queue
2759:         if self.grammar_manager.has_waiting_grammars():
      # 注：分支判断：根据请求的采样/grammar 约束 `self.grammar_manager.has_waiting_grammars()` 决定是否进入受限解码路径。
2760:             ready_grammar_requests = self.grammar_manager.get_ready_grammar_requests()
      # 注：局部状态绑定：`ready_grammar_requests` 保存请求或 batch 状态，后续调度/执行会继续改写它。
2761:             for req in ready_grammar_requests:
      # 注：循环处理：通常是在遍历请求、rank、worker、token 或候选项。
2762:                 self._add_request_to_queue(req)
      # 注：成员函数调用：`self._add_request_to_queue` 会读取或更新当前对象状态，建议继续查看该方法定义。
2763: 
2764:         if self.enable_hierarchical_cache:
      # 注：分支判断：根据缓存/内存资源 `self.enable_hierarchical_cache` 决定是否复用、加载、驱逐或降级。
2765:             self.tree_cache.check_hicache_events()
      # 注：成员函数调用：`self.tree_cache.check_hicache_events` 会读取或更新当前对象状态，建议继续查看该方法定义。
2766: 
2767:         if self.enable_priority_preemption or self.is_hybrid_swa:
      # 注：分支判断：只有满足 `self.enable_priority_preemption or self.is_hybrid_swa` 时才进入该路径；这通常是在区分部署模式、请求类型或资源状态。
2768:             # Reset batch_is_full to try preemption with a prefill adder.
2769:             self.running_batch.batch_is_full = False
      # 注：成员变量写入：`self.running_batch.batch_is_full` 会留在对象生命周期中，后续方法可能依赖这份状态。
2770: 
2771:         if (
      # 注：多行分支开始：完整条件在接下来几行，通常用于组合多个请求参数或运行时状态。
2772:             self.running_batch.batch_is_full or len(self.waiting_queue) == 0
2773:         ) and self.chunked_req is None:
2774:             return None
      # 注：返回出口：把本阶段整理出的状态交给调用方；读调用链时要回到上层看它如何被消费。
2775: 
2776:         running_bs = len(self.running_batch.reqs)
      # 注：局部状态绑定：`running_bs` 保存本阶段的中间结果，后续几行通常会立即消费它。
2777:         if self._should_delay_dflash_prefill_for_batching(running_bs):
      # 注：分支判断：根据请求/batch 状态 `self._should_delay_dflash_prefill_for_batching(running_bs)` 决定调度、执行或回包路径。
2778:             return None
      # 注：返回出口：把本阶段整理出的状态交给调用方；读调用链时要回到上层看它如何被消费。
2779: 
2780:         # Ignore the check if self.chunked_req is not None.
2781:         # In the non-PP case, when self.chunked_req is not None, num_allocatable_reqs should always be greater than 0,
2782:         # as the space for the chunked requests has just been released.
2783:         # In PP case, chunked requests (or dllm requests) can start in one microbatch and end in another microbatch, so the max_running_requests per microbatch should not be strict.
2784:         # Instead, we should always allow chunked requests to be added, otherwise, there will be a memory leak.
2785:         if (
      # 注：多行分支开始：完整条件在接下来几行，通常用于组合多个请求参数或运行时状态。
2786:             self.get_num_allocatable_reqs(running_bs) <= 0
      # 注：成员函数调用：`self.get_num_allocatable_reqs` 会读取或更新当前对象状态，建议继续查看该方法定义。
2787:             and self.chunked_req is None
2788:             and not self.enable_priority_preemption
2789:         ):
2790:             self.running_batch.batch_is_full = True
      # 注：成员变量写入：`self.running_batch.batch_is_full` 会留在对象生命周期中，后续方法可能依赖这份状态。
2791:             return None
      # 注：返回出口：把本阶段整理出的状态交给调用方；读调用链时要回到上层看它如何被消费。
2792: 
2793:         # Get priority queue
2794:         self.policy.calc_priority(self.waiting_queue, self.running_batch)
      # 注：成员函数调用：`self.policy.calc_priority` 会读取或更新当前对象状态，建议继续查看该方法定义。
2795: 
```

**读这段时抓住：**

- 它不是简单 pop waiting queue，而要同时检查 grammar、memory、chunked prefill、disaggregation、priority 等条件。
- 这里形成的是 prefill/extend batch；decode 的 running batch 在另一条路径继续推进。
- 如果你新增调度策略，通常要看它在这段之前还是之后改变 waiting queue。


### `run_batch`：scheduler 和 model worker 的窄腰

```python
# python/sglang/srt/managers/scheduler.py:3178-3238
3178:     def run_batch(
      # 注：函数定义：`run_batch` 是调用边界；参数决定它从上游接收哪些状态，返回值决定下游能看到什么。
3179:         self,
3180:         batch: ScheduleBatch,
3181:         pp_proxy_tensors: Optional[PPProxyTensors] = None,
3182:     ) -> Union[GenerationBatchResult, EmbeddingBatchResult]:
3183:         """Run a batch."""
3184:         self.forward_ct += 1
      # 注：成员变量写入：`self.forward_ct +` 会留在对象生命周期中，后续方法可能依赖这份状态。
3185:         batch.forward_iter = self.forward_ct
      # 注：局部状态绑定：`batch.forward_iter` 保存请求或 batch 状态，后续调度/执行会继续改写它。
3186: 
3187:         if self.scripted_scheduler_hook is not None:
      # 注：分支判断：只有满足 `self.scripted_scheduler_hook is not None` 时才进入该路径；这通常是在区分部署模式、请求类型或资源状态。
3188:             self.scripted_scheduler_hook.on_run_batch(batch)
      # 注：成员函数调用：`self.scripted_scheduler_hook.on_run_batch` 会读取或更新当前对象状态，建议继续查看该方法定义。
3189: 
3190:         # Whether to run the profiler
3191:         self.profiler_manager._profile_batch_predicate(batch)
      # 注：成员函数调用：`self.profiler_manager._profile_batch_predicate` 会读取或更新当前对象状态，建议继续查看该方法定义。
3192:         if self.forward_sleep_time is not None:
      # 注：分支判断：只有满足 `self.forward_sleep_time is not None` 时才进入该路径；这通常是在区分部署模式、请求类型或资源状态。
3193:             logger.info(f"Scheduler.run_batch sleep {self.forward_sleep_time}s")
      # 注：对象/库方法调用：`logger.info` 把当前对象状态交给另一个组件处理，建议追踪该对象的生命周期。
3194:             time.sleep(self.forward_sleep_time)
      # 注：对象/库方法调用：`time.sleep` 把当前对象状态交给另一个组件处理，建议追踪该对象的生命周期。
3195: 
3196:         # Place holder handling for pd-disagg decode event loop
3197:         if batch.forward_mode.is_prebuilt():
      # 注：分支判断：根据请求/batch 状态 `batch.forward_mode.is_prebuilt()` 决定调度、执行或回包路径。
3198:             return self._run_batch_prebuilt(batch)
      # 注：返回出口：把本阶段整理出的状态交给调用方；读调用链时要回到上层看它如何被消费。
3199: 
3200:         # Run forward
3201:         if self.is_generation:
      # 注：分支判断：只有满足 `self.is_generation` 时才进入该路径；这通常是在区分部署模式、请求类型或资源状态。
3202:             if self.enable_overlap:
      # 注：分支判断：只有满足 `self.enable_overlap` 时才进入该路径；这通常是在区分部署模式、请求类型或资源状态。
3203:                 # Self-gates on batch.spec_info.future_indices; non-spec_v2
3204:                 # no-ops (ForwardBatch.init_new lazily computes the sum).
3205:                 self.future_map.resolve_seq_lens_cpu(batch)
      # 注：成员函数调用：`self.future_map.resolve_seq_lens_cpu` 会读取或更新当前对象状态，建议继续查看该方法定义。
3206: 
3207:                 with self.forward_stream_ctx:
3208:                     self.forward_stream.wait_stream(self.schedule_stream)
      # 注：成员函数调用：`self.forward_stream.wait_stream` 会读取或更新当前对象状态，建议继续查看该方法定义。
3209:                     # resolve consumes SB staging (prefill_input_ids_cpu /
3210:                     # mix_running_indices). Run OUTSIDE isolation so the
3211:                     # snapshot captures the post-consume state — restoring
3212:                     # post-forward must not un-consume staging.
3213:                     resolve_forward_inputs(batch, self.future_map)
      # 注：函数/库调用：`resolve_forward_inputs` 把当前阶段委托给外部 helper 或库实现。
3214: 
3215:                     with self._forward_isolation(batch, overlap=True):
      # 注：局部状态绑定：`with self._forward_isolation(batch, overlap` 保存请求或 batch 状态，后续调度/执行会继续改写它。
3216:                         future_indices = batch.req_pool_indices
      # 注：局部状态绑定：`future_indices` 保存异步任务句柄，后续会检查完成、取消或取结果。
3217: 
3218:                         # Spec_v2 fires on_publish mid-worker (between verify and
3219:                         # draft_extend) so schedule prep can overlap with draft_extend.
3220:                         # Non-spec has no later work — scheduler publishes after return.
3221:                         fwd_kwargs = (
      # 注：局部状态绑定：`fwd_kwargs` 保存配置快照，下面的分支通常会基于它选择执行路径。
3222:                             {
3223:                                 "on_publish": partial(
      # 注：函数/库调用：`partial` 把当前阶段委托给外部 helper 或库实现。
3224:                                     self.future_map.publish, future_indices
3225:                                 )
3226:                             }
3227:                             if not batch.spec_algorithm.is_none()
      # 注：分支判断：根据请求/batch 状态 `not batch.spec_algorithm.is_none()` 决定调度、执行或回包路径。
3228:                             else {}
3229:                         )
3230: 
3231:                         # FIXME: pp is not compatible with overlap
3232:                         batch_result = self.model_worker.forward_batch_generation(
      # 注：局部状态绑定：`batch_result` 保存请求或 batch 状态，后续调度/执行会继续改写它。
3233:                             batch, **fwd_kwargs
3234:                         )
3235:                         if batch.spec_algorithm.is_none():
      # 注：分支判断：根据请求/batch 状态 `batch.spec_algorithm.is_none()` 决定调度、执行或回包路径。
3236:                             self.future_map.publish(future_indices, batch.seq_lens + 1)
      # 注：成员函数调用：`self.future_map.publish` 会读取或更新当前对象状态，建议继续查看该方法定义。
3237:                         # Park any refs the worker wants kept alive 2 iters
3238:                         # (cross-stream tensor lifetime; pinned in the same
```

**读这段时抓住：**

- `run_batch` 接收 `ScheduleBatch`，根据模式走普通 forward、speculative forward、embedding/score 等路径。
- 它是 GPU 执行的调用点，但仍在 scheduler 进程里负责前后状态 bookkeeping。
- 很多性能 Feature 会在这里附近插入计时、profile、overlap 或特殊 forward mode。


## ModelRunner：从 batch 到模型 forward

`ModelRunner` 是权重、attention backend、CUDA graph runner、forward context 的聚合点。新增模型时通常在 `srt/models` 实现模型结构；新增执行优化时通常会碰 `model_executor` 或 attention backend。


In [ ]:
for name in ["ModelRunner", "forward", "forward_decode", "forward_extend"]:
    rows = find_defs("python/sglang/srt/model_executor/model_runner.py", {name})
    print(name, rows[:5])


## 小练习：用源码检查 Scheduler 主循环有哪些旁路 Feature

这个 cell 抓取 `Scheduler.__init__` 附近常见子系统名。它不是完整解析器，但能帮助你建立“Feature 都挂在哪里”的直觉。


In [ ]:
scheduler_src = read_rel("python/sglang/srt/managers/scheduler.py")
keywords = [
    "grammar", "spec", "lora", "disaggregation", "hicache",
    "overlap", "cuda_graph", "metrics", "profile", "cache_controller",
]
for kw in keywords:
    hits = [i+1 for i, line in enumerate(scheduler_src.splitlines()) if kw in line.lower()]
    print(f"{kw:18s}", hits[:10], "..." if len(hits) > 10 else "")


## 贡献者注意点

- Scheduler 里的 bug 往往是状态生命周期 bug：请求 abort、streaming、chunked prefill、grammar pending、spec verify 都可能改变同一个 batch。
- KV 分配失败不一定是 OOM，也可能是 prefix cache 锁住了太多节点或 page reserve 估计不对。
- 改采样参数时要同时检查 tokenizer manager 的输入校验、scheduler 的 batch 合并、sampling batch info 的 tensor 化。
